# Custom MoleculeNet Dataset (PyTorch)

The `MoleculeNetDataset` class is intended for datasets that consist of a table of SMILES and corresponding targets,
converting them into tensor representations for graph networks. The class provides properties and methods for making
graph features from SMILES using RDKit.

The graph structure matches the molecular graph (chemical bonds). Features for atoms and bonds are generated with
RDKit. Atomic coordinates are generated by a conformer guess and cached in an SDF file.

This tutorial demonstrates:
1. Creating a custom `MoleculeNetDataset` subclass
2. SMILES parsing and molecular graph construction
3. Feature engineering for molecular graphs
4. Converting to PyG Data objects
5. Training a GNN model with PyTorch

## 0. Create Example Data

For demonstration, we make an artificial table of SMILES and some target values.

In [ ]:
import os
import numpy as np

os.makedirs("ExampleMol", exist_ok=True)
csv_data = "\n".join([
    "smiles,Values1,Values2",
    "CCC, 1, 0.1",
    "CCCO, 2, 0.3",
    "CCCN, 3, 0.2",
    "CCCC=O, 4, 0.4",
    "NOCF, 4, 1.4",
])
with open("ExampleMol/data.csv", "w") as f:
    f.write(csv_data)

The file structure expected:

```
ExampleMol/
    data.csv
    data.sdf   # Created by prepare_data()
```

## 1. Initialization

To load the dataset, `MoleculeNetDataset` requires the data directory path, the CSV file name, and a dataset name.

In [ ]:
from kgcnn_torch.data.moleculenet import MoleculeNetDataset, OneHotEncoder

dts = MoleculeNetDataset(
    file_name="data.csv",
    data_directory="ExampleMol/",
    dataset_name="ExampleMol"
)

## 2. Data Preparation

Precompute the molecular structure (and optionally 3D coordinates) from SMILES and cache as an SDF file.
The SDF file is stored in the same directory. Structure generation can run in parallel.

In [ ]:
dts.prepare_data(
    overwrite=True,
    smiles_column_name="smiles",
    add_hydrogen=True,
    make_conformers=True,
    optimize_conformer=True,
    num_workers=None
)

## 3. Read Data and Set Attributes

After `prepare_data()` is called, molecules are read from the cached SDF file.
We specify which node, edge, and graph-level attributes to extract,
along with encoders (e.g., one-hot encoding for atomic symbols).

In [ ]:
# Inspect available attribute identifiers
from kgcnn_torch.molecule.graph_rdkit import MolecularGraphRDKit
mol = MolecularGraphRDKit()
print("Atom attributes:", list(mol.atom_fun_dict.keys())[:10], "...")
print("Bond attributes:", list(mol.bond_fun_dict.keys())[:10], "...")
print("Molecule attributes:", list(mol.mol_fun_dict.keys())[:10], "...")

In [ ]:
# Custom callback: extract the number of atoms as a graph-level feature
def graph_size_callback(mg, ds):
    return mg.mol.GetNumAtoms()

# Custom transform: compute Gasteiger partial charges
def custom_trafo(mg):
    return mg.compute_partial_charges()

In [ ]:
dts.set_attributes(
    label_column_name=["Values1", "Values2"],
    # Node attributes
    nodes=["Symbol", "TotalNumHs", "GasteigerCharge"],
    encoder_nodes={
        "Symbol": OneHotEncoder(["C", "N", "O"], dtype="str", add_unknown=False)
    },
    # Edge attributes
    edges=["BondType", "Stereo"],
    encoder_edges={
        "BondType": int
    },
    # Graph-level attributes
    graph=["ExactMolWt"],
    additional_callbacks={"size": graph_size_callback},
    custom_transform=custom_trafo,
    add_hydrogen=False,
    has_conformers=True
)
print("Number of graphs:", len(dts))

## 4. Inspecting Graphs

In [ ]:
# Check node symbols and atomic numbers
print("Node symbols:", dts.obtain_property("node_symbol"))
print("Node numbers:", dts.obtain_property("node_number"))

In [ ]:
# Print a single graph
print(dts[3])

In [ ]:
# Inspect computed attributes
print("Node attributes (graph 0):", dts.obtain_property("node_attributes")[0])
print("Edge attributes (graph 0):", dts.obtain_property("edge_attributes")[0])
print("Graph attributes:", dts.obtain_property("graph_attributes"))
print("Graph labels:", dts.obtain_property("graph_labels"))

In [ ]:
# Visualize one molecular graph using networkx
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_nodes_from([
    (i, {"atom": x})
    for i, x in enumerate(dts.obtain_property("node_symbol")[3])
])
G.add_edges_from(dts.obtain_property("edge_indices")[3])
labels = nx.get_node_attributes(G, "atom")
nx.draw(G, labels=labels, with_labels=True, node_color="lightblue")
plt.title("Molecular graph for CCCC=O")
plt.show()

## 5. Convert to PyG and Train a Model

The `to_pyg_list()` method converts our `MemoryGraphList` to a list of PyG `Data` objects.
We then use a standard PyTorch training loop with `kgcnn_torch.training.trainer.fit()`.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

# Convert to PyG Data objects
pyg_list = dts.to_pyg_list(
    node_key="node_number",
    pos_key="node_coordinates",
    edge_key="edge_indices",
    label_key="graph_labels"
)

# Ensure multi-target labels are 2D (1, num_targets) per graph so PyG
# stacks them to (B, num_targets) instead of concatenating 1D to (B*K,).
for d in pyg_list:
    if d.y is not None and d.y.dim() == 1 and d.y.numel() > 1:
        d.y = d.y.unsqueeze(0)

print(f"Number of PyG graphs: {len(pyg_list)}")
print(f"Example graph: {pyg_list[0]}")

In [ ]:
# Build a SchNet model for property prediction
from kgcnn_torch.models.schnet import SchNetModel

model = SchNetModel(
    node_dim=32,
    depth=3,
    units=64,
    gauss_bins=20,
    gauss_distance=4.0,
    gauss_sigma=0.4,
    last_mlp_units=[64, 32],
    num_targets=2,          # We predict Values1 and Values2
    output_embedding="graph",
    make_distance=True,
    expand_distance=True
)
print(model)

In [ ]:
# Simple train/test split (small dataset, just for demonstration)
train_idx, test_idx = train_test_split(
    np.arange(len(pyg_list)), test_size=0.4, random_state=42
)
train_data = [pyg_list[i] for i in train_idx]
test_data = [pyg_list[i] for i in test_idx]

train_loader = DataLoader(train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(test_data, batch_size=2)

In [ ]:
from kgcnn_torch.training.trainer import fit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    loss_fn=nn.MSELoss(),
    epochs=50,
    device=device,
    metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))},
    verbose=1
)

print(f"Final train loss: {history['train_loss'][-1]:.4f}")
if history['val_loss']:
    print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## 6. Save and Load Dataset

The dataset can be serialized to disk and loaded later without re-parsing SMILES.

In [ ]:
dts.save("ExampleMol/ExampleMol.kgcnn.pickle")
print("Saved dataset.")

# Reload
dts_loaded = MoleculeNetDataset(
    file_name="data.csv",
    data_directory="ExampleMol/",
    dataset_name="ExampleMol"
)
dts_loaded.load("ExampleMol/ExampleMol.kgcnn.pickle")
print("Loaded dataset, length:", len(dts_loaded))
print(dts_loaded[0])

## Summary

This notebook demonstrated the full workflow for using `MoleculeNetDataset` in kgcnn-torch:

1. **Initialization** -- Point to a CSV of SMILES + labels
2. **prepare_data()** -- Convert SMILES to 3D structures (SDF)
3. **set_attributes()** -- Extract node/edge/graph features with RDKit
4. **to_pyg_list()** -- Convert to PyG Data objects
5. **fit()** -- Train a PyTorch GNN model using the standard trainer